# Visual Place Recognition — start your project

Notebook per Colab. Le sezioni 1-5 sono la pipeline VPR di base; la parte
**Adaptive Re-ranking** e' l'estensione del progetto.

Se l'image matching top-20 e' gia' stato calcolato e salvato su Drive,
salta direttamente alla sezione **A. Test rapido**: rilegge quei risultati e
gira in pochi minuti su CPU, senza rifare nessun matching.

## Setup

In [ ]:
!git clone --recursive https://github.com/tommasopantano01/Visual-Place-Recognition-Project
%cd Visual-Place-Recognition-Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Le dipendenze pesanti (`image-matching-models`) servono **solo** per eseguire
davvero l'image matching. Per validation e per l'adaptive re-ranking in
modalita' offline bastano numpy / pandas / scikit-learn / torch, gia' presenti
su Colab.

In [ ]:
# solo se devi calcolare l'image matching (serve GPU)
%cd image-matching-models
!pip install -e .[all]
!pip install faiss-cpu
%cd ..

In [ ]:
!python download.py   # scarica i dataset

## 1. Retrieval

`--save_for_uncertainty` deve restare attivo: produce `z_data.torch` con le
distanze L2, necessarie ai metodi SU.

In [ ]:
!python VPR-methods-evaluation/main.py \
--num_workers 8 \
--batch_size 32 \
--log_dir log_dir \
--method=cosplace --backbone=ResNet18 --descriptors_dimension=512 \
--image_size 512 512 \
--database_folder '<path-to-database-folder>' \
--queries_folder '<path-to-queries-folder>' \
--num_preds_to_save 20 \
--recall_values 1 5 10 20 \
--save_for_uncertainty

## 2. Image matching sui risultati del retrieval (serve GPU)

In [ ]:
!python match_queries_preds.py \
--preds-dir '<path-to-predictions-folder>' \
--matcher 'superpoint-lg' \
--device 'cuda' \
--num-preds 20

## 3. Full re-ranking (baseline)

Stampa la recall del solo retrieval (BASE) e quella dopo il re-ranking completo
del top-20. E' il riferimento a costo massimo contro cui si misura l'adaptive.

In [ ]:
!python reranking.py \
--preds-dir '<path-to-predictions-folder>' \
--inliers-dir '<path-to-inliers-folder>' \
--num-preds 20 \
--recall-values 1 5 10 20

## 4. Uncertainty evaluation [solo studenti AML]

In [ ]:
!python -m vpr_uncertainty.eval \
--preds-dir '<path-to-predictions-folder>' \
--inliers-dir '<path-to-inliers-folder>' \
--z-data-path '<path-to-z-data-file>'

---
# Adaptive Re-ranking

Il full re-ranking fa image matching su tutti i 20 candidati di ogni query.
L'adaptive decide **query per query** se quel costo vale la pena. Quasi tutti i
metodi pagano la decisione con un solo IM sul top-1; `su` decide dalle sole
distanze del retrieval e costa **zero** IM.

Metodi (`--threshold`): `youden`, `best_r1`, `efficiency` (nessun training),
`logistic_hard`, `logistic_help`, `logistic_cost_sensitive`, `sequential`
(servono i regressori allenati), `su`, `su_inliers` (servono anche le distanze L2).
`local` non e' ancora implementato.

## A. Test rapido — riusa l'image matching gia' calcolato

`--inliers-dir` rilegge i `.torch` top-20 gia' presenti su Drive invece di
rifare il matching: **stessi numeri, pochi minuti, nessuna GPU**.

Il comando qui sotto esegue **tutti i metodi su tutte le coppie (model, matcher)**
e scrive una tabella riassuntiva in `<output-root>/summary_deploy.csv`.
Le combinazioni con file mancanti vengono saltate ed elencate alla fine, quindi
si puo' lanciare anche a dati parziali.

Adatta i tre template ai tuoi path reali: `{model}`, `{matcher}` e `{dataset}`
vengono sostituiti automaticamente (i wildcard `*` sono ammessi).

In [ ]:
# controlla che i template puntino davvero a qualcosa PRIMA di lanciare
from glob import glob
DATASET = 'svox_training'
T_PREDS   = '/content/drive/MyDrive/VPR/preds/{model}_{dataset}'
T_INLIERS = '/content/drive/MyDrive/VPR/matching_results/{model}_{dataset}_{matcher}'
T_ZDATA   = '/content/drive/MyDrive/VPR/logs/{model}_{dataset}/z_data.torch'

for model in ('cosplace', 'megaloc'):
    for matcher in ('superpoint-lg', 'loftr'):
        for name, t in (('preds', T_PREDS), ('inliers', T_INLIERS), ('z_data', T_ZDATA)):
            hits = glob(t.format(model=model, matcher=matcher, dataset=DATASET))
            print(f"{model:9} {matcher:14} {name:8} {len(hits)} -> {hits[:1]}")

In [ ]:
!python VPR-Adaptive-ReRanking/run_all_methods.py \
--preds-dir-template   '/content/drive/MyDrive/VPR/preds/{model}_{dataset}' \
--inliers-dir-template '/content/drive/MyDrive/VPR/matching_results/{model}_{dataset}_{matcher}' \
--z-data-template      '/content/drive/MyDrive/VPR/logs/{model}_{dataset}/z_data.torch' \
--dataset 'svox_training' \
--output-root '/content/drive/MyDrive/VPR/adaptive_test'

In [ ]:
import pandas as pd
pd.read_csv('/content/drive/MyDrive/VPR/adaptive_test/summary_deploy.csv')

## B. Un singolo metodo

Stessi argomenti, un metodo alla volta. Togliendo `--inliers-dir` il matching
viene eseguito live (serve GPU e `image-matching-models`).

In [ ]:
!python VPR-Adaptive-ReRanking/adaptive_reranking.py \
--threshold 'logistic_help' \
--preds-dir '<path-to-predictions-folder>' \
--model 'cosplace' --matcher 'superpoint-lg' \
--inliers-dir '<path-to-top20-inliers-folder>' \
--output-dir '<path-to-output-folder>' \
--num-preds 20
# su / su_inliers vogliono anche:  --z-data '<path-to-z_data.torch>'

In [ ]:
!python VPR-Adaptive-ReRanking/check_performance.py \
--preds-dir '<path-to-predictions-folder>' \
--adaptive-RR-dir '<path-to-output-folder>' \
--num-preds 20 \
--recall-values 1 5 10 20

## C. Ricalibrare da zero (training + validation)

Serve solo se cambi split o dataset. Il **candidate-level CSV** va costruito due
volte, su split diversi: quello di train per il training, quello di validation
per la scelta delle soglie.

In [ ]:
# training dei regressori SU (scrive model_*.json in validation/<features>/)
!python VPR-Adaptive-ReRanking/train_su.py \
--features 'su' \
--train-csv '<path-to-train-candidate_level.csv>' \
--model 'cosplace' --matcher 'superpoint-lg'

In [ ]:
# oppure scarica i regressori gia' allenati
!pip install gdown
!python VPR-Adaptive-ReRanking/validation/download_models.py \
--zip '/content/drive/MyDrive/VPR/validation_models.zip'

In [ ]:
# validation: sceglie le soglie e scrive threshold_<model>_<matcher>.csv
!python VPR-Adaptive-ReRanking/validation/run_all.py \
--val-csv-template '/content/drive/MyDrive/VPR/candidate_level/val_{model}_{matcher}.csv'

In [ ]:
import pandas as pd
pd.read_csv('VPR-Adaptive-ReRanking/validation/summary.csv')